In [2]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
from GBM import simulate_GBM, simulate_GBM_antithetic, simulate_GBM_numba_parallel, simulate_GBM_numba
from black_scholes import Black_Scholes, bs_delta, bs_vega
from european import european_mc, european_mc_antithetic, european_mc_control_variate
from exotic import asian_mc, geometric_asian_bs, price_lookback_mc, barrier_mc
from greeks import pathwise_delta, pathwise_vega, lr_delta, lr_vega, fd_delta, fd_vega
from american import american_longstaff_schwartz, american_binomial, american_longstaff_schwartz_fast

import matplotlib.pyplot as plt
from scipy.stats import qmc

import cProfile
import numba
from numba import njit, prange

In [3]:
N = 1_000_000
n_steps = 252

start = time.perf_counter()
t, S = simulate_GBM(100, 0.05, 0.2, 1.0, n_steps, N)
elapsed = time.perf_counter() - start
print(f"NumPy vectorized: {elapsed:.3f} sec for {N:,} paths")

NumPy vectorized: 20.926 sec for 1,000,000 paths


In [4]:
cProfile.run('simulate_GBM(100, 0.05, 0.2, 1.0, 252, 1_000_000)', sort='cumulative')

         249 function calls in 19.539 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    0.000    0.000   29.538    7.385 base_events.py:1970(_run_once)
        3    0.000    0.000   10.005    3.335 selectors.py:310(select)
        3    7.693    2.564   10.005    3.335 selectors.py:304(_select)
        1    0.000    0.000    9.528    9.528 {built-in method builtins.exec}
        1    0.862    0.862    9.528    9.528 <string>:1(<module>)
        1    7.787    7.787    8.666    8.666 <string>:1(simulate_GBM)
        1    0.000    0.000    2.186    2.186 fromnumeric.py:2879(cumsum)
        1    0.000    0.000    2.186    2.186 fromnumeric.py:51(_wrapfunc)
        1    2.186    2.186    2.186    2.186 {method 'cumsum' of 'numpy.ndarray' objects}
        1    0.879    0.879    0.879    0.879 shape_base.py:295(hstack)
        3    0.126    0.042    0.126    0.042 {built-in method select.select}
        2    0.005    0

In [5]:
# warmup call to trigger JIT compilation, not counted
_ = simulate_GBM_numba(100, 0.05, 0.2, 1.0, 10, 10)

N_values = [10_000, 100_000, 1_000_000]
numpy_times = []
numba_times = []

for N in N_values:
    start = time.perf_counter()
    simulate_GBM(100, 0.05, 0.2, 1.0, 252, N)
    numpy_times.append(time.perf_counter() - start)
    
    start = time.perf_counter()
    simulate_GBM_numba(100, 0.05, 0.2, 1.0, 252, N)
    numba_times.append(time.perf_counter() - start)

for N, t_np, t_nb in zip(N_values, numpy_times, numba_times):
    print(f"N={N:>9,}: NumPy {t_np:.3f}s, Numba {t_nb:.3f}s, speedup {t_np/t_nb:.1f}x")

N=   10,000: NumPy 0.253s, Numba 0.263s, speedup 1.0x
N=  100,000: NumPy 2.244s, Numba 2.720s, speedup 0.8x
N=1,000,000: NumPy 23.571s, Numba 25.977s, speedup 0.9x


In [6]:
_ = simulate_GBM_numba_parallel(100, 0.05, 0.2, 1.0, 10, 10)  # warmup

parallel_times = []
for N in N_values:
    start = time.perf_counter()
    simulate_GBM_numba_parallel(100, 0.05, 0.2, 1.0, 252, N)
    parallel_times.append(time.perf_counter() - start)

for N, t_np, t_nb, t_par in zip(N_values, numpy_times, numba_times, parallel_times):
    print(f"N={N:>9,}: NumPy {t_np:.3f}s, Numba {t_nb:.3f}s, Parallel {t_par:.3f}s, speedup vs NumPy: {t_np/t_par:.1f}x")

N=   10,000: NumPy 0.253s, Numba 0.263s, Parallel 0.060s, speedup vs NumPy: 4.2x
N=  100,000: NumPy 2.244s, Numba 2.720s, Parallel 0.857s, speedup vs NumPy: 2.6x
N=1,000,000: NumPy 23.571s, Numba 25.977s, Parallel 8.391s, speedup vs NumPy: 2.8x


In [7]:
print(f"Numba sees {numba.config.NUMBA_NUM_THREADS} threads available")

Numba sees 28 threads available


In [8]:
for n_steps_test in [20, 252]:
    start = time.perf_counter()
    simulate_GBM_numba_parallel(100, 0.05, 0.2, 1.0, n_steps_test, 1_000_000)
    t_par = time.perf_counter() - start
    start = time.perf_counter()
    simulate_GBM(100, 0.05, 0.2, 1.0, n_steps_test, 1_000_000)
    t_np = time.perf_counter() - start
    print(f"n_steps={n_steps_test}: NumPy {t_np:.3f}s, Parallel {t_par:.3f}s, speedup {t_np/t_par:.1f}x")

n_steps=20: NumPy 2.050s, Parallel 0.708s, speedup 2.9x
n_steps=252: NumPy 23.943s, Parallel 7.656s, speedup 3.1x


In [9]:
def benchmark(fn, *args, n_repeats=5):
    times = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - start)
    times = np.array(times)
    return times.mean(), times.std()

N = 1_000_000
n_steps = 252

mean_np, std_np = benchmark(simulate_GBM, 100, 0.05, 0.2, 1.0, n_steps, N)
mean_par, std_par = benchmark(simulate_GBM_numba_parallel, 100, 0.05, 0.2, 1.0, n_steps, N)

print(f"NumPy:    {mean_np:.3f} ± {std_np:.3f}s")
print(f"Parallel: {mean_par:.3f} ± {std_par:.3f}s")
print(f"Speedup:  {mean_np/mean_par:.1f}x (using means)")


NumPy:    23.659 ± 0.140s
Parallel: 7.768 ± 0.118s
Speedup:  3.0x (using means)


In [10]:
cProfile.run('american_longstaff_schwartz(100, 100, 0.05, 0.2, 1.0, 252, 100000, type="put")', sort='cumulative')

         16021 function calls in 4.810 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    4.497    4.497 {built-in method builtins.exec}
        1    0.018    0.018    4.497    4.497 <string>:1(<module>)
        1    2.402    2.402    4.479    4.479 <string>:1(american_longstaff_schwartz)
      251    1.082    0.004    1.101    0.004 _linalg.py:2442(lstsq)
        1    0.477    0.477    0.477    0.477 <string>:1(simulate_GBM)
      252    0.262    0.001    0.262    0.001 <string>:6(<lambda>)
        1    0.000    0.000    0.200    0.200 fromnumeric.py:2879(cumsum)
        1    0.000    0.000    0.200    0.200 fromnumeric.py:51(_wrapfunc)
        1    0.200    0.200    0.200    0.200 {method 'cumsum' of 'numpy.ndarray' objects}
      251    0.165    0.001    0.171    0.001 shape_base.py:220(vstack)
        1    0.106    0.106    0.106    0.106 shape_base.py:295(hstack)
      251    0.002    0.00

In [11]:
def test_ls_fast_matches_original():
    price_orig, se_orig = american_longstaff_schwartz(100, 100, 0.05, 0.2, 1.0, 252, 100000, "put")
    price_fast, se_fast = american_longstaff_schwartz_fast(100, 100, 0.05, 0.2, 1.0, 252, 100000, "put")
    assert abs(price_orig - price_fast) < 3 * max(se_orig, se_fast)

In [12]:
mean_before, std_before = benchmark(american_longstaff_schwartz, 100, 100, 0.05, 0.2, 1.0, 252, 100000, "put", n_repeats=5)
mean_after, std_after = benchmark(american_longstaff_schwartz_fast, 100, 100, 0.05, 0.2, 1.0, 252, 100000, "put", n_repeats=5)

print(f"Before: {mean_before:.3f} ± {std_before:.3f}s")
print(f"After:  {mean_after:.3f} ± {std_after:.3f}s")
print(f"Speedup: {mean_before/mean_after:.2f}x")

Before: 6.525 ± 0.278s
After:  4.011 ± 0.100s
Speedup: 1.63x


In [13]:
price_before, se_before = american_longstaff_schwartz(100, 100, 0.05, 0.2, 1.0, 252, 100000, "put")
price_after, se_after = american_longstaff_schwartz_fast(100, 100, 0.05, 0.2, 1.0, 252, 100000, "put")
print(f"Before: {price_before:.4f} ± {1.96*se_before:.4f}")
print(f"After:  {price_after:.4f} ± {1.96*se_after:.4f}")

Before: 6.0207 ± 0.0431
After:  6.0795 ± 0.0441


In [14]:
test_ls_fast_matches_original()